# *QOT Estimator Basic Usage Example*

> This notebook demonstrates the basic usage of the QOT estimator package for optical link.

In [1]:
# Import Dependencies
import sys
from pathlib import Path
import os
import numpy as np
import pandas as pd
import json
from scipy.io import loadmat

# Ignore warnings to have clean cell outputs
import warnings
warnings.filterwarnings("ignore")

### *Create results directory if it doesn't exist*

In [2]:
# Get the current working directory (where the notebook is running)
base_dir = Path.cwd()

# Define the results directory
results_dir = base_dir.parent / "results" / "single_link"

# Create it if it doesn't exist
results_dir.mkdir(exist_ok=True)

print(f"Results will be saved in: {results_dir}")

Results will be saved in: d:\Projects\CoreLight\Code\CFM_versions\CFM\results\single_link


In [3]:
sys.path.append(os.path.abspath(base_dir.parent / 'src'))
from CFM.core.network import Link, LinkParameters
from CFM.core.band import Band, OpticalParameters
from CFM.core.qot_estimator import ParameterBuilder, ISRSSolver, NLISolver, ASESolver, OSNRCalculator, MultiCore_Parameters, OSNRCalculatorV2
from CFM.core.qot_optmizer import BasePowerModel, PowerOptimizer, FLPPowerModel, FRPPowerModel, PowerOptimizerV2
from CFM.core.post_process import GSNRPlotter
from CFM.utils.build_alpha_db import build_alpha_for_band
from CFM.utils.SRS_effect import SRS_effect

In [4]:
# Get the current working directory (where the notebook is running)
base_dir = Path.cwd()

# Define the results directory
results_dir = base_dir.parent / "results" / "single_link"

# Create it if it doesn't exist
results_dir.mkdir(exist_ok=True)

print(f"Results will be saved in: {results_dir}")

Results will be saved in: d:\Projects\CoreLight\Code\CFM_versions\CFM\results\single_link


### *Create fiber link*

In [5]:
# my_link_params = LinkParameters()
# my_link_l1 = Link(
#   name='l1',
#   length=70,
#   num_span=1,
#   num_amp=1,
#   link_params=my_link_params)

### *Online QOT Estimation*

In [6]:
# c = 3e8 # Speed of light in m/s
# channel_spacing_granularity_thz = 12.5e9 / 1e12 # 0.0125 THz
# channel_spacing_thz = 12 * channel_spacing_granularity_thz # 0.150 THz
# n_guard_channel=3
# # Optical Parameters (assuming the same Rs_mat as your example)
# band_params = OpticalParameters(Rs_mat=120*1e9)

# # --- L-Band ---
# # Wavelengths: 1626 nm to 1575.4 nm
# start_freq_l = (c / (1626 * 1e-9)) / 1e12
# end_freq_l = (c / (1575.4 * 1e-9)) / 1e12

# my_band_L = Band(
#     name='L',
#     start_freq=start_freq_l,
#     end_freq=end_freq_l,
#     opt_params=band_params,
#     channel_spacing=channel_spacing_thz
# )
# spectrum_my_L = my_band_L.calc_spectrum() + channel_spacing_thz * 0.5
# guard_L_C = spectrum_my_L[0] + np.arange(1, n_guard_channel + 1) * channel_spacing_thz

# my_Gband_L_C = Band(
#     name='G',
#     start_freq=spectrum_my_L[0]+channel_spacing_thz * 0.5,
#     end_freq=spectrum_my_L[0] + n_guard_channel * channel_spacing_thz,
#     opt_params=band_params,
#     channel_spacing=channel_spacing_thz
# )
# spectrum_my_G_L_C = my_Gband_L_C.calc_spectrum() + channel_spacing_thz * 0.5

# # --- C-Band ---
# # Wavelengths: 1572.1 nm to 1524.1 nm
# start_freq_c = guard_L_C[-1] + channel_spacing_thz / 2
# end_freq_c = (c / (1524.1 * 1e-9)) / 1e12

# my_band_C = Band(
#     name='C',
#     start_freq=start_freq_c,
#     end_freq=end_freq_c,
#     opt_params=band_params,
#     channel_spacing=channel_spacing_thz
# )
# spectrum_my_C = my_band_C.calc_spectrum() + channel_spacing_thz * 0.5
# guard_C_S = spectrum_my_C[0] + np.arange(1, n_guard_channel + 1) * channel_spacing_thz
# my_Gband_C_S = Band(
#     name='G',
#     start_freq=spectrum_my_C[0]+channel_spacing_thz * 0.5,
#     end_freq=spectrum_my_C[0] + n_guard_channel * channel_spacing_thz,
#     opt_params=band_params,
#     channel_spacing=channel_spacing_thz
# )
# spectrum_my_G_C_S = my_Gband_C_S.calc_spectrum() + channel_spacing_thz * 0.5
# # --- S-Band ---
# # Wavelengths: 1520 nm to 1460 nm
# start_freq_s = guard_C_S[-1] + channel_spacing_thz / 2
# end_freq_s = (c / (1460 * 1e-9)) / 1e12

# my_band_S = Band(
#     name='S',
#     start_freq=start_freq_s,
#     end_freq=end_freq_s,
#     opt_params=band_params,
#     channel_spacing=channel_spacing_thz
# )
# spectrum_my_S = my_band_S.calc_spectrum() + channel_spacing_thz * 0.5
# guard_S_E = spectrum_my_S[0] + np.arange(1, n_guard_channel + 1) * channel_spacing_thz
# my_Gband_S_E = Band(
#     name='G',
#     start_freq=spectrum_my_S[0]+channel_spacing_thz * 0.5,
#     end_freq=spectrum_my_S[0] + n_guard_channel * channel_spacing_thz,
#     opt_params=band_params,
#     channel_spacing=channel_spacing_thz
# )
# spectrum_my_G_S_E = my_Gband_S_E.calc_spectrum() + channel_spacing_thz * 0.5
# # --- E-Band ---
# # Wavelengths: 1456 nm to 1360 nm
# start_freq_e = guard_S_E[-1] + channel_spacing_thz / 2
# end_freq_e = (c / (1360 * 1e-9)) / 1e12

# my_band_E = Band(
#     name='E',
#     start_freq=start_freq_e,
#     end_freq=end_freq_e,
#     opt_params=band_params,
#     channel_spacing=channel_spacing_thz
# )
# spectrum_my_E = my_band_E.calc_spectrum() + channel_spacing_thz * 0.5

# # --- Calculate Channel Counts (Optional, if you still need the exact numbers) ---
# n_guard_channel = round((0.4 * 1e12) / (channel_spacing_thz * 1e12))
# n_channel_L = round((end_freq_l - start_freq_l) / channel_spacing_thz) + 1
# n_channel_C = round((end_freq_c - start_freq_c) / channel_spacing_thz)
# n_channel_S = round((end_freq_s - start_freq_s) / channel_spacing_thz)
# n_channel_E = round((end_freq_e - start_freq_e) / channel_spacing_thz)

# n_all_channels = (n_channel_L + n_guard_channel + n_channel_C + n_guard_channel + 
#                   n_channel_S + n_guard_channel + n_channel_E)





# grid_center = np.concatenate((
#     spectrum_my_L[::-1], 
#     spectrum_my_G_L_C[::-1], 
#     spectrum_my_C[::-1], 
#     spectrum_my_G_C_S[::-1], 
#     spectrum_my_S[::-1], 
#     spectrum_my_G_S_E[::-1], 
#     spectrum_my_E[::-1]
# ))
# grid_center = grid_center*1e12
# bands = [my_band_L, my_Gband_L_C, my_band_C, my_Gband_C_S, my_band_S, my_Gband_S_E, my_band_E]


In [7]:
# P_in = -2.5 # dBm

# # Calculate powers in Watts
# idle_power_W = 10 ** ((-80 - 30) / 10)  # -80 dBm
# active_power_W = 10 ** ((P_in - 30) / 10) # P_in dBm

# # Initialize arrays with the idle power (-80 dBm)
# # Note: n_channel_L, n_channel_C, etc., should be integer types
# launch_power_L = np.full(int(n_channel_L), idle_power_W)
# launch_power_C = np.full(int(n_channel_C), idle_power_W)
# launch_power_S = np.full(int(n_channel_S), idle_power_W)
# launch_power_E = np.full(int(n_channel_E), idle_power_W)
# launch_power_guard = np.full(int(n_guard_channel), idle_power_W)

# # Link states (0-based indexing for Python)
# # np.arange(start, stop, step) 
# link_state_L = np.arange(0, 30, 2)
# link_state_C = np.arange(0, 30, 2)
# link_state_S = np.arange(0, 50, 2)
# link_state_E = np.arange(0, 90, 2)

# # Assign active power to the specific link state channels
# launch_power_L[link_state_L] = active_power_W
# launch_power_C[link_state_C] = active_power_W
# launch_power_S[link_state_S] = active_power_W
# launch_power_E[link_state_E] = active_power_W

# # launching power = initial condition for solving ODE
# y0 = np.concatenate((
#     launch_power_L, 
#     launch_power_guard, 
#     launch_power_C, 
#     launch_power_guard, 
#     launch_power_S, 
#     launch_power_guard, 
#     launch_power_E
# ))

In [8]:
# alpha_L   = build_alpha_for_band(None, None, band=my_band_L)
# alpha_L_C   = build_alpha_for_band(None, None, band=my_Gband_L_C)
# alpha_C   = build_alpha_for_band(None, None, band=my_band_C)
# alpha_C_S   = build_alpha_for_band(None, None, band=my_Gband_C_S)
# alpha_S  = build_alpha_for_band(None, None, band=my_band_S)
# alpha_S_E   = build_alpha_for_band(None, None, band=my_Gband_S_E)
# alpha_E  = build_alpha_for_band(None, None, band=my_band_E)

# alpha_dB_LCS = np.hstack([alpha_L, alpha_L_C, alpha_C, alpha_C_S, alpha_S, alpha_S_E, alpha_E])


In [9]:
PHI = 241*[2/3]

In [10]:
# my_link_Online_SNR_Param = ParameterBuilder(
#   link=my_link_l1,
#   bands=bands,
#   P_in = y0,
#   grid_center=grid_center,
#   alpha_dB_LCS=alpha_dB_LCS,
#   PHI = PHI
# )

In [11]:
# my_link_ISRS = ISRSSolver(my_link_Online_SNR_Param, 'Online')

In [12]:
# s1, s2, sig = my_link_ISRS.solve_Online()

In [13]:
# my_link_NLI = NLISolver(my_link_Online_SNR_Param, s1, s2, sig)

In [14]:
# my_link_NLI.solve_online()

### *QOT Estimation*

### *Create spectrum*

In [15]:
# my_band_l_params =  OpticalParameters(Rs_mat = 52e9)
# my_band_l = Band(
#     name='l',
#     start_freq = (1.845393470967740e+2 - 0.075/2), # THz
#     end_freq = (1.904643470967740e+2 + 0.075/2), # THz
#     opt_params = my_band_l_params,
#     channel_spacing = 0.075 # THz
#     )
# spectrum_my_l = my_band_l.calc_spectrum() + 0.075*0.5

# my_band_c_params =  OpticalParameters(Rs_mat = 52e9)
# my_band_c = Band(
#     name='c',
#     start_freq = (1.909143470967740e+2 - 0.075/2), # THz
#     end_freq = (1.968393470967740e+2 + 0.075/2), # THz
#     opt_params = my_band_c_params,
#     channel_spacing = 0.075 # THz
#     )
# spectrum_my_c = my_band_c.calc_spectrum() + 0.075*0.5

# my_band_s_part1_params =  OpticalParameters(Rs_mat = 52e9)
# my_band_s_part1 = Band(
#     name='s',
#     start_freq = (1.972893470967740e+2 - 0.075/2), # THz
#     end_freq = (2.035143470967740e+2 + 0.075/2), # THz
#     opt_params = my_band_s_part1_params,
#     channel_spacing = 0.075 # THz
#     )
# spectrum_my_s_part1 = my_band_s_part1.calc_spectrum() + 0.075*0.5


# my_band_s_part2_params =  OpticalParameters(Rs_mat = 52e9)
# my_band_s_part2 = Band(
#     name='s',
#     start_freq = (2.035893870967740e+2 - 0.075/2), # THz
#     end_freq = (2.053143870967740e+2 + 0.075/2), # THz
#     opt_params = my_band_s_part2_params,
#     channel_spacing = 0.075 # THz
#     )
# spectrum_my_s_part2 = my_band_s_part2.calc_spectrum() + 0.075*0.5

# grid_center = np.concatenate((spectrum_my_s_part2 , spectrum_my_s_part1, spectrum_my_c , spectrum_my_l))
# grid_center = grid_center[::-1]*1e12
# bands = [my_band_l, my_band_c, my_band_s_part1, my_band_s_part2]

In [16]:
alpha_dB_LCS_268channels = loadmat('.././data/alpha_dB_LCS_268channels.mat')
alpha_dB_LCS = alpha_dB_LCS_268channels['alpha_dB_LCS']

In [17]:
# alpha_L   = build_alpha_for_band(None, None, band=my_band_l)
# alpha_C   = build_alpha_for_band(None, None, band=my_band_c)
# alpha_S1  = build_alpha_for_band(None, None, band=my_band_s_part1)
# alpha_S2  = build_alpha_for_band(None, None, band=my_band_s_part2)

# alpha_dB_LCS1 = np.hstack([alpha_L, alpha_C, alpha_S1, alpha_S2])


In [18]:
my_band_c_params =  OpticalParameters(Rs_mat = 52e9)
my_band_c = Band(
    name='c',
    start_freq = (1.909143470967740e+2), # THz
    end_freq = (1.968393470967750e+2), # THz
    opt_params = my_band_c_params,
    channel_spacing = 0.075 # THz
    )
spectrum_my_c = my_band_c.calc_spectrum() 
grid_center = spectrum_my_c[::-1]*1e12
bands = [my_band_c]

In [19]:
alpha_C   = build_alpha_for_band(None, None, band=my_band_c)

alpha_dB_C = np.hstack([alpha_C])


In [20]:
len(grid_center)

80

In [21]:
grid_center

array([1.90914347e+14, 1.90989347e+14, 1.91064347e+14, 1.91139347e+14,
       1.91214347e+14, 1.91289347e+14, 1.91364347e+14, 1.91439347e+14,
       1.91514347e+14, 1.91589347e+14, 1.91664347e+14, 1.91739347e+14,
       1.91814347e+14, 1.91889347e+14, 1.91964347e+14, 1.92039347e+14,
       1.92114347e+14, 1.92189347e+14, 1.92264347e+14, 1.92339347e+14,
       1.92414347e+14, 1.92489347e+14, 1.92564347e+14, 1.92639347e+14,
       1.92714347e+14, 1.92789347e+14, 1.92864347e+14, 1.92939347e+14,
       1.93014347e+14, 1.93089347e+14, 1.93164347e+14, 1.93239347e+14,
       1.93314347e+14, 1.93389347e+14, 1.93464347e+14, 1.93539347e+14,
       1.93614347e+14, 1.93689347e+14, 1.93764347e+14, 1.93839347e+14,
       1.93914347e+14, 1.93989347e+14, 1.94064347e+14, 1.94139347e+14,
       1.94214347e+14, 1.94289347e+14, 1.94364347e+14, 1.94439347e+14,
       1.94514347e+14, 1.94589347e+14, 1.94664347e+14, 1.94739347e+14,
       1.94814347e+14, 1.94889347e+14, 1.94964347e+14, 1.95039347e+14,
      

#### *Multo Span*

In [22]:
from CFM.core.qot_estimator import MultiSpanSystem

In [23]:
my_link_params = LinkParameters()
my_link_l1 = Link(
  name='l1',
  length=[70],
  num_span=1,
  num_amp=1,
  link_params=my_link_params)

In [24]:
cfm_system = MultiSpanSystem(
    link=my_link_l1,
    bands=[my_band_c],
    P_in=-2.0,
    grid_center=spectrum_my_c[::-1]*1e12,
    isrs_model='FLP',
    alpha_dB_LCS=alpha_C
)

In [25]:
# cfm_system.set_input_power(-2.0)
# results = cfm_system.run()
# final_span_osnr = results["OSNR_NLI_ASE_dB"][-1, :]

In [26]:
# # Power sweep range: -2.0 to 2.0 dBm in 0.1 increments
# p_in_sweep = np.arange(-2.0, 2.1, 0.1)

# # Store results
# osnr_nli_ase_results = []

# for p_in in p_in_sweep:
#     print(f"Simulating for input power: {p_in:.1f} dBm")
    
#     # Update power and run the pipeline
#     cfm_system.set_input_power(p_in)
#     results = cfm_system.run()
    
#     # Extract the final combined OSNR array for the last span
#     # results["OSNR_NLI_ASE_dB"] has shape (N_ss, N_c), we want the final span [-1]
#     final_span_osnr = results["OSNR_NLI_ASE_dB"][-1, :]
#     osnr_nli_ase_results.append(final_span_osnr)

# # Convert to numpy array for easy plotting/analysis
# osnr_nli_ase_results = np.array(osnr_nli_ase_results)

#### *FRP Example*

In [27]:
my_link_params = LinkParameters()
my_link_l1 = Link(
  name='l1',
  length=[70, 70],
  num_span=2,
  num_amp=1,
  link_params=my_link_params)

In [28]:
my_link_SNR_Param = ParameterBuilder(
  link=my_link_l1,
  bands=[my_band_c],
  P_in = -2.,
  grid_center=spectrum_my_c[::-1]*1e12,
  alpha_dB_LCS=alpha_C
)

In [29]:
my_link_ISRS = ISRSSolver(my_link_SNR_Param, 'FLP')

In [30]:
p, s1, s2, sig = my_link_ISRS.solve()

In [31]:
s1

array([[2.16871317e-05, 2.16953593e-05, 2.17037194e-05, 2.17121995e-05,
        2.17208092e-05, 2.17295458e-05, 2.17384050e-05, 2.17473898e-05,
        2.17565005e-05, 2.17657416e-05, 2.17751051e-05, 2.17845882e-05,
        2.17946474e-05, 2.18049129e-05, 2.18154459e-05, 2.18262156e-05,
        2.18372072e-05, 2.18484099e-05, 2.18598663e-05, 2.18715084e-05,
        2.18833418e-05, 2.18953937e-05, 2.19076234e-05, 2.19200467e-05,
        2.19326716e-05, 2.19454768e-05, 2.19584325e-05, 2.19715671e-05,
        2.19848616e-05, 2.19982536e-05, 2.20118840e-05, 2.20256686e-05,
        2.20395941e-05, 2.20536829e-05, 2.20541831e-05, 2.20708553e-05,
        2.20831428e-05, 2.20967552e-05, 2.21112075e-05, 2.21305406e-05,
        2.21508003e-05, 2.21399335e-05, 2.21290564e-05, 2.21183603e-05,
        2.21724447e-05, 2.21774774e-05, 2.21825693e-05, 2.21877915e-05,
        2.22363622e-05, 2.22471333e-05, 2.22580435e-05, 2.22691666e-05,
        2.22803914e-05, 2.22917639e-05, 2.23033000e-05, 2.231497

In [32]:
my_link_ISRS = ISRSSolver(my_link_SNR_Param, 'FLP')

In [33]:
my_link_NLI = NLISolver(my_link_SNR_Param, s1, s2, sig)

In [35]:
nli = my_link_NLI.solve()

In [36]:
my_link_ASE = ASESolver(my_link_SNR_Param) 

In [37]:
ase = my_link_ASE.solve()

In [38]:
my_link_OSNR = OSNRCalculator(my_link_SNR_Param, ase, nli)

In [39]:
on, oa, ot = my_link_OSNR.compute()

In [40]:
ot

array([[26.78717831, 26.77404912, 26.76501868, 26.75725001, 26.75007065,
        26.7432352 , 26.73662928, 26.73017367, 26.72382336, 26.71753743,
        26.71131465, 26.70514773, 26.69860903, 26.69202871, 26.68534918,
        26.67858427, 26.67173869, 26.66483048, 26.65778573, 26.65070408,
        26.64356187, 26.63631351, 26.62898889, 26.62161308, 26.61414192,
        26.60659659, 26.59900985, 26.59134643, 26.58362639, 26.57594638,
        26.56811607, 26.56021986, 26.55227918, 26.54430866, 26.5362924 ,
        26.52827066, 26.52015976, 26.51201546, 26.50382201, 26.49560996,
        26.48733149, 26.47904421, 26.47069729, 26.4623219 , 26.45390482,
        26.4455035 , 26.43703289, 26.42853732, 26.41999134, 26.41141848,
        26.40284477, 26.39435881, 26.38575398, 26.37714002, 26.36854041,
        26.35991886, 26.35131338, 26.34274984, 26.33417498, 26.32563099,
        26.31715217, 26.30868966, 26.3002183 , 26.29186374, 26.28354471,
        26.27529334, 26.26712438, 26.25906362, 26.2

In [28]:
curves = [
    {"name": "GSNR_total", "values": ot, "color": "blue"},
    {"name": "GSNR_linear", "values": oa, "color": "green"},
    {"name": "GSNR_nonlinear", "values": on, "color": "red"},
]

plotter = GSNRPlotter(my_link_SNR_Param, curves, bands=[my_band_c])

fig = plotter.plot(N_s_max=1, matlab_indexing=True)
fig.show()


In [29]:
my_link_OSNRV2 = OSNRCalculatorV2(my_link_SNR_Param, ase, nli)

In [30]:
res = my_link_OSNRV2.compute()

In [31]:
res['OSNR_NLI_ASE_dB']

array([[31.24461329, 31.22109935, 31.2083366 , 31.19883757, 31.19092152,
        31.1839357 , 31.17755854, 31.17160698, 31.16596703, 31.16056228,
        31.15533986, 31.15025762, 31.14526917, 31.140364  , 31.13552549,
        31.1307409 , 31.126     , 31.12129458, 31.11661447, 31.11195832,
        31.10732012, 31.10269301, 31.09807448, 31.09346325, 31.08885442,
        31.08424672, 31.07963997, 31.07503143, 31.07042148, 31.06581293,
        31.06119537, 31.05657075, 31.05194092, 31.04730729, 31.04266834,
        31.03802323, 31.03337335, 31.02871865, 31.02405828, 31.01939477,
        31.01472682, 31.01005762, 31.00538756, 31.0007161 , 30.99604272,
        30.99137379, 30.98671095, 30.98205075, 30.9773978 , 30.97275649,
        30.96813325, 30.96353953, 30.95895875, 30.9544029 , 30.9498795 ,
        30.94539267, 30.94095115, 30.93656369, 30.93223602, 30.92797882,
        30.92380465, 30.91972243, 30.9157453 , 30.91189643, 30.90819143,
        30.90465424, 30.90131406, 30.89820665, 30.8

#### *FLP Example*

In [52]:
my_link_SNR_Param = ParameterBuilder(
  link=my_link_l1,
  bands=bands,
  P_in = 2.,
  grid_center=grid_center,
  alpha_dB_LCS=alpha_dB_LCS
)

In [53]:
my_link_ISRS = ISRSSolver(my_link_SNR_Param, 'FLP')

In [54]:
s1, s2, sig = my_link_ISRS.solve()

In [55]:
my_link_NLI = NLISolver(my_link_SNR_Param, s1, s2, sig)

In [56]:
nli = my_link_NLI.solve()

In [57]:
my_link_ASE = ASESolver(my_link_SNR_Param) 

In [58]:
ase = my_link_ASE.solve()

In [59]:
my_link_OSNR = OSNRCalculator(my_link_SNR_Param, ase, nli)

In [60]:
on, oa, ot = my_link_OSNR.compute()

In [61]:
curves = [
    {"name": "GSNR_total", "values": ot, "color": "blue"},
    {"name": "GSNR_linear", "values": oa, "color": "green"},
    {"name": "GSNR_nonlinear", "values": on, "color": "red"},
]

plotter = GSNRPlotter(my_link_SNR_Param, curves, bands=bands)

fig = plotter.plot(N_s_max=1, matlab_indexing=True)
fig.show()


In [62]:
ot

array([[32.78196928, 32.37875334, 32.19855991, 32.0827865 , 31.99766468,
        31.93042838, 31.87485255, 31.82740224, 31.7856893 , 31.74566866,
        31.71271161, 31.68357484, 31.65755617, 31.63418412, 31.61301624,
        31.59379313, 31.57627913, 31.56028529, 31.54565672, 31.53222751,
        31.51983345, 31.50833735, 31.49756155, 31.48700663, 31.47349483,
        31.46445004, 31.45661797, 31.44954371, 31.44309694, 31.43725206,
        31.43191119, 31.42708021, 31.4226912 , 31.4186705 , 31.41490513,
        31.41118211, 31.40597874, 31.4027126 , 31.39984338, 31.3971641 ,
        31.39452788, 31.39195793, 31.38935701, 31.38673872, 31.38404929,
        31.38133606, 31.37845024, 31.37520109, 31.36988984, 31.36660458,
        31.36366173, 31.36092809, 31.35813401, 31.35537932, 31.35270508,
        31.35006435, 31.34743827, 31.34491756, 31.342257  , 31.33883129,
        31.33658067, 31.33471199, 31.33306739, 31.33180867, 31.33079372,
        31.32973297, 31.32969555, 31.33040484, 31.3

### *QOT Power Optimizer*

#### *FRP Example*

In [41]:
model = FRPPowerModel()

In [42]:
my_link_params = LinkParameters()
my_link_l1 = Link(
  name='l1',
  length=30.3,
  num_span=1,
  num_amp=1,
  link_params=my_link_params)

In [43]:
P_optimizer = PowerOptimizerV2(
  link=my_link_l1,
  bands=bands,
  grid_center=grid_center,
  model=model,
  base='OSNR_NLI_ASE_dB',
  alpha_dB_LCS=alpha_dB_LCS
)

In [44]:
res = P_optimizer.optimize()

In [45]:
res

{'P_opt': np.float64(-7.0600000000000005),
 'gsnr': array([[38.31392107, 38.25112398, 38.21934709, 38.19699048, 38.17922764,
         38.16417774, 38.15091671, 38.13892135, 38.12785719, 38.11750924,
         38.1077228 , 38.09838282, 38.08938298, 38.08067159, 38.07220115,
         38.06392463, 38.05580819, 38.04782293, 38.03993114, 38.03209815,
         38.02421347, 38.01482879, 38.00706582, 37.99949672, 37.99200848,
         37.98456738, 37.97715979, 37.96977181, 37.96238613, 37.9550012 ,
         37.94760127, 37.94017732, 37.93272698, 37.9252475 , 37.91772943,
         37.91017777, 37.90257994, 37.8948756 , 37.88616737, 37.87845467,
         37.87083576, 37.86325827, 37.85570125, 37.84816008, 37.84063251,
         37.83311705, 37.82560886, 37.81810547, 37.81060183, 37.80309528,
         37.79558713, 37.78808114, 37.78056087, 37.77302155, 37.76545666,
         37.75783746, 37.75005837, 37.74064168, 37.73284402, 37.72519247,
         37.71758273, 37.70998905, 37.70240631, 37.69484104, 

In [ ]:
# curves = [
#     {"name": "GSNR_total", "values": res["gsnr"], "color": "blue"},
#     {"name": "GSNR_linear", "values": res["osnr"], "color": "green"},
#     {"name": "GSNR_nonlinear", "values": res["snr_nli"], "color": "red"},
# ]

# plotter = GSNRPlotter(my_link_SNR_Param, curves)

# fig = plotter.plot(N_s_max=1, matlab_indexing=True)
# fig.show()


#### *FLP Example*

In [46]:
model = FLPPowerModel()

In [47]:
P_optimizer = PowerOptimizer(
  link=my_link_l1,
  bands=bands,
  grid_center=grid_center,
  model=model,
  alpha_dB_LCS=alpha_dB_LCS
)

In [ ]:
# res = P_optimizer.optimize()

In [ ]:
# res

### *Multi Core Fiber QOT Estimation*

In [ ]:
my_link_SNR_Param = ParameterBuilder(
  link=my_link_l1,
  bands=bands,
  P_in = -15,
  grid_center=grid_center,
  alpha_dB_LCS=alpha_dB_LCS
)

In [ ]:
my_link_mcf_params = MultiCore_Parameters(
  link=my_link_l1,
  grid_center=grid_center,
  N_c=my_link_SNR_Param.N_c,
  core_pitch = 43*10**-6,
  adjacent_core_vec =[3, 3, 4, 3, 3]
)

In [ ]:
my_link_ISRS = ISRSSolver(my_link_SNR_Param, 'FRP')

In [ ]:
s1, s2, sig = my_link_ISRS.solve()

In [ ]:
my_link_NLI = NLISolver(my_link_SNR_Param, s1, s2, sig)

In [ ]:
nli = my_link_NLI.solve()

In [ ]:
my_link_ASE = ASESolver(my_link_SNR_Param) 

In [ ]:
ase = my_link_ASE.solve()

In [ ]:
my_link_OSNRV2 = OSNRCalculatorV2(my_link_SNR_Param, ase, nli, my_link_mcf_params)

In [ ]:
res = my_link_OSNRV2.compute()

In [ ]:
res['OSNR_NLI_ASE_XT_dB']

array([[17.08893995, 17.18077003, 17.27265717, ..., 27.75941564,
        27.82730453, 28.00237625],
       [17.08893995, 17.18077003, 17.27265717, ..., 27.75941564,
        27.82730453, 28.00237625],
       [15.88414906, 15.97703229, 16.06995863, ..., 27.72277217,
        27.79089823, 27.9653071 ],
       [17.08893995, 17.18077003, 17.27265717, ..., 27.75941564,
        27.82730453, 28.00237625],
       [17.08893995, 17.18077003, 17.27265717, ..., 27.75941564,
        27.82730453, 28.00237625]])

In [48]:
model = FRPPowerModel()

In [49]:
P_optimizer = PowerOptimizerV2(
  link=my_link_l1,
  bands=bands,
  grid_center=grid_center,
  model=model,
  base='OSNR_NLI_ASE_XT_dB',
  MCF_params=my_link_mcf_params,
  alpha_dB_LCS=alpha_dB_LCS
)

NameError: name 'my_link_mcf_params' is not defined

In [ ]:
P_optimizer.optimize()

{'P_opt': np.float64(-14.1),
 'gsnr': array([[17.11906299, 17.21140078, 17.30387542, ..., 27.00463214,
         27.12049831, 27.43000444],
        [17.11906299, 17.21140078, 17.30387542, ..., 27.00463214,
         27.12049831, 27.43000444],
        [15.90694355, 16.0002158 , 16.09359195, ..., 26.97379772,
         27.08952531, 27.39748158],
        [17.11906299, 17.21140078, 17.30387542, ..., 27.00463214,
         27.12049831, 27.43000444],
        [17.11906299, 17.21140078, 17.30387542, ..., 27.00463214,
         27.12049831, 27.43000444]]),
 'osnr': array([[31.98154954, 31.97977617, 31.97800774, 31.97624145, 31.97447654,
         31.97271273, 31.97094989, 31.96918794, 31.96742683, 31.96566654,
         31.96390704, 31.9621483 , 31.96039032, 31.95863308, 31.95687659,
         31.95512082, 31.95336577, 31.95161145, 31.94985784, 31.94810495,
         31.94635277, 31.94460129, 31.94285052, 31.94110045, 31.93935108,
         31.93760241, 31.93585444, 31.93410715, 31.93236056, 31.93061465,

Learning Packages

In [72]:
import joblib
base_dir = Path.cwd()
data = joblib.load(base_dir.parent/"data"/"LC_model_package.pkl")

model = data["model"]
Tx = data["x_scaler"]
Ty = data["y_scaler"]


WindowsPath('d:/Projects/CoreLight/Code/CFM_versions/CFM')